In [1]:
import os  # added 2026: credentials were scrubbed to environment reads
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingRegressor, StackingClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression, Ridge 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import pykalman
import joblib
import gc
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import sys
import json
from config import ALLOWED_ELEMENT_TYPES,ICON_COLOR_MAP
from utils import reformat_scraped_data
from webdriver_manager.chrome import ChromeDriverManager
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning) 
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus1_tz = pytz.timezone('Etc/GMT-2')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+1
        localized_time = utc_plus1_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df
    
def news_fetch():
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        driver = webdriver.Chrome()
    except:
        print ("AF: No Chrome webdriver installed")
        driver = webdriver.Chrome(ChromeDriverManager().install())

    driver.get("https://www.forexfactory.com/calendar")

    month =  datetime.now().strftime("%B")

    table = driver.find_element(By.CLASS_NAME, "calendar__table")

    data = []
    previous_row_count = 0
    # Scroll down to the end of the page
    while True:
        # Record the current scroll position
        before_scroll = driver.execute_script("return window.pageYOffset;")
        
        # Scroll down a fixed amount
        driver.execute_script("window.scrollTo(0, window.pageYOffset + 500);")
        
        # Wait for a short moment to allow content to load
        time.sleep(2)
        
        # Record the new scroll position
        after_scroll = driver.execute_script("return window.pageYOffset;")
        
        # If the scroll position hasn't changed, we've reached the end of the page
        if before_scroll == after_scroll:
            break

    # Now that we've scrolled to the end, collect the data
    for row in table.find_elements(By.TAG_NAME, "tr"):
        row_data = []
        for element in row.find_elements(By.TAG_NAME, "td"):
            class_name = element.get_attribute('class')
            if class_name in ALLOWED_ELEMENT_TYPES:
                if element.text:
                    row_data.append(element.text)
                elif "calendar__impact" in class_name:
                    impact_elements = element.find_elements(By.TAG_NAME, "span")
                    for impact in impact_elements:
                        impact_class = impact.get_attribute("class")
                        color = ICON_COLOR_MAP[impact_class]
                    if color:
                        row_data.append(color)
                    else:
                        row_data.append("impact")

        if len(row_data):
            data.append(row_data)

    reformat_scraped_data(data,month)
    ds = pd.read_csv(f'{month}_news.csv', parse_dates=["date"], index_col=0)
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    convert_time_to_mt5(ds)
    return ds

def get_signal():
    ticker = 'XAUUSD_i'
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 777)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])
        
    # Calculate technical indicators
    df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
    df["max"] = df['Close'].rolling(21).max() / df['Close'] - 1
    df["min"] = df ['Close'].rolling(21).min() / df['Close'] - 1
    df["boll"] = (df['Close'] - df['Close'].rolling(21).mean()) / df['Close'].rolling(21).std()
    df.dropna(inplace=True)
    
    # STL
    stl = STL(df['returns'], seasonal=21, period=21).fit()
    df['stl_trend'] = stl.trend
    df['stl_seasonal'] = stl.seasonal
    df['stl_resid'] = stl.resid
    df.dropna(inplace=True)

    features = ['stl_trend', 'stl_resid', 'max', 'min', 'boll']
    X = df[features]
    objects = joblib.load('U2.joblib')
    scaler_clf = objects['scaler_clf']
    X = scaler_clf.transform(X)
    X = pd.DataFrame(X,columns=features)
    model = objects['stacking_clf']
    df['pred'] = model.predict(X)
    
    signal = df.pred.values[-1]
    signal_time = df.index[-1]
    
    return signal, signal_time

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    print(f"Close order result: {result}")


def execute_trade(signal, qty):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")

def get_mt5_time():
    tz_mt5 = pytz.timezone('Etc/GMT-3')  # Use the correct timezone
    now = datetime.now(tz_mt5)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    

def get_next_bar_time(interval):
    now = get_mt5_time()
    now = datetime.strptime(now, '%Y-%m-%d %H:%M:%S')
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")


def check_and_trade(stop_event):

    while not stop_event.is_set():
        try:         
            # Print mt5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting
                       
            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes= 30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}.previous was opposite closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

# Create a stop event
stop_event = Event()

# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

True

In [ ]:
try:
    # Start the trading loop
    check_and_trade(stop_event)
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")

Current MT5 time: 2024-09-18 23:02:56
Next check time: 2024-09-18 23:15:01
------------------------------------------------------------------------
New signal checked: 1.0, Signal time: 2024-09-18 23:00:00
Current signal is 1.0.previous was opposite closing position.
Close order result: OrderSendResult(retcode=10009, deal=191368863, order=226522443, volume=1.0, price=2555.14, bid=0.0, ask=0.0, comment='Request executed', request_id=1769333775, retcode_external=0, request=TradeRequest(action=1, magic=0, order=0, symbol='XAUUSD_i', volume=1.0, price=2555.14, stoplimit=0.0, sl=0.0, tp=0.0, deviation=0, type=0, type_filling=1, type_time=0, expiration=0, comment='close the position', position=226495627, position_by=0))
Executing BUY order: XAUUSD_i, Volume: 1.0, Price: 2555.14
Trade order result: OrderSendResult(retcode=10009, deal=191368865, order=226522445, volume=1.0, price=2555.14, bid=0.0, ask=0.0, comment='Request executed', request_id=1769333776, retcode_external=0, request=TradeRequ

In [ ]:
import time
from datetime import datetime, timedelta

def check_and_trade(stop_event, news_df):
    while not stop_event.is_set():
        try:
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Check for upcoming news events
            news_times = news_df['time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
            for news_time in news_times:
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time}. Halting trading.")
                    time.sleep(60)  # Wait before checking again
                    continue  # Skip further processing

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error


In [ ]:
try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    news_df = pd.read_csv(file_name)

    # Start the trading loop
    check_and_trade(stop_event, news_df)
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")


In [ ]:

# Create the filename
filename = f"{month}_news.csv"

# Sample data to write to the CSV (replace this with your actual data)
data = {'example_column': [ds]}  # Replace 'example_column' with your actual column name
df = pd.DataFrame(data)

# Write the DataFrame to a CSV file
df.to_csv(filename, index=False)

In [ ]:
import pandas as pd
import threading
from datetime import datetime, timedelta
import MetaTrader5 as mt5

# Define your stop event
stop_event = threading.Event()

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus2_tz = pytz.timezone('Etc/GMT-2')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+2
        localized_time = utc_plus2_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df

def check_and_trade(stop_event, news_df):
    while not stop_event.is_set():
        try:
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Filter news events for the current day
            current_date = MT5.date()
            news_df_today = news_df[news_df['date'].dt.date == current_date]
            
            # Check for upcoming news events
            news_times = news_df_today['time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
            for news_time in news_times:
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time}. Halting trading.")
                    time.sleep(60)  # Wait before checking again
                    continue  # Skip further processing

            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting
                       
            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    ds = pd.read_csv(file_name, parse_dates=["date"], index_col=0)
    
    # Filter the news DataFrame
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    
    # Convert the times to MT5 time zone
    ds = convert_time_to_mt5(ds)

    # Start the trading loop
    check_and_trade(stop_event, ds)
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingRegressor, StackingClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression, Ridge 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import pykalman
import joblib
import gc
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import sys
import json
from config import ALLOWED_ELEMENT_TYPES,ICON_COLOR_MAP
from utils import reformat_scraped_data
from webdriver_manager.chrome import ChromeDriverManager
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning) 
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus1_tz = pytz.timezone('Etc/GMT-2')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+1
        localized_time = utc_plus1_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df
    
def news_fetch():
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        driver = webdriver.Chrome()
    except:
        print ("AF: No Chrome webdriver installed")
        driver = webdriver.Chrome(ChromeDriverManager().install())

    driver.get("https://www.forexfactory.com/calendar")

    month =  datetime.now().strftime("%B")

    table = driver.find_element(By.CLASS_NAME, "calendar__table")

    data = []
    previous_row_count = 0
    # Scroll down to the end of the page
    while True:
        # Record the current scroll position
        before_scroll = driver.execute_script("return window.pageYOffset;")
        
        # Scroll down a fixed amount
        driver.execute_script("window.scrollTo(0, window.pageYOffset + 500);")
        
        # Wait for a short moment to allow content to load
        time.sleep(2)
        
        # Record the new scroll position
        after_scroll = driver.execute_script("return window.pageYOffset;")
        
        # If the scroll position hasn't changed, we've reached the end of the page
        if before_scroll == after_scroll:
            break

    # Now that we've scrolled to the end, collect the data
    for row in table.find_elements(By.TAG_NAME, "tr"):
        row_data = []
        for element in row.find_elements(By.TAG_NAME, "td"):
            class_name = element.get_attribute('class')
            if class_name in ALLOWED_ELEMENT_TYPES:
                if element.text:
                    row_data.append(element.text)
                elif "calendar__impact" in class_name:
                    impact_elements = element.find_elements(By.TAG_NAME, "span")
                    for impact in impact_elements:
                        impact_class = impact.get_attribute("class")
                        color = ICON_COLOR_MAP[impact_class]
                    if color:
                        row_data.append(color)
                    else:
                        row_data.append("impact")

        if len(row_data):
            data.append(row_data)

    reformat_scraped_data(data,month)
    ds = pd.read_csv(f'{month}_news.csv', parse_dates=["date"], index_col=0)
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    convert_time_to_mt5(ds)
    filename = f"{month}_news.csv"
    ds.to_csv(filename, index=False)
    return ds

def get_signal():
    ticker = 'XAUUSD_i'
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 777)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])
        
    # Calculate technical indicators
    df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
    df["max"] = df['Close'].rolling(21).max() / df['Close'] - 1
    df["min"] = df ['Close'].rolling(21).min() / df['Close'] - 1
    df["boll"] = (df['Close'] - df['Close'].rolling(21).mean()) / df['Close'].rolling(21).std()
    df.dropna(inplace=True)
    
    # STL
    stl = STL(df['returns'], seasonal=21, period=21).fit()
    df['stl_trend'] = stl.trend
    df['stl_seasonal'] = stl.seasonal
    df['stl_resid'] = stl.resid
    df.dropna(inplace=True)

    features = ['stl_trend', 'stl_resid', 'max', 'min', 'boll']
    X = df[features]
    objects = joblib.load('U2.joblib')
    scaler_clf = objects['scaler_clf']
    X = scaler_clf.transform(X)
    X = pd.DataFrame(X,columns=features)
    model = objects['stacking_clf']
    df['pred'] = model.predict(X)
    
    signal = df.pred.values[-1]
    signal_time = df.index[-1]
    
    return signal, signal_time

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    print(f"Close order result: {result}")


def execute_trade(signal, qty):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")

def get_mt5_time():
    tz_mt5 = pytz.timezone('Etc/GMT-3')  # Use the correct timezone
    now = datetime.now(tz_mt5)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    

def get_next_bar_time(interval):
    now = get_mt5_time()
    now = datetime.strptime(now, '%Y-%m-%d %H:%M:%S')
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")


def check_and_trade(stop_event, news_df):
    while not stop_event.is_set():
        try:
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Filter news events for the current day
            current_date = MT5.date()
            news_df_today = news_df[news_df['date'].dt.date == current_date]
            
            # Check for upcoming news events
            news_times = news_df_today['time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
            for news_time in news_times:
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time}. Halting trading.")
                    time.sleep(60)  # Wait before checking again
                    continue  # Skip further processing
                    
            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

# Create a stop event
stop_event = Event()

# get the red news
news_fetch()

# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

In [ ]:
try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    news_df = pd.read_csv(file_name)

    # Start the trading loop
    check_and_trade(stop_event, news_df)
    
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")

In [ ]:
import schedule
import time
from datetime import datetime
import pytz

# Define your function
def your_function():
    print("Function called at", datetime.now(pytz.timezone('Etc/GMT-3')))

# Schedule the function to run every day at midnight in the specified timezone
def schedule_job():
    timezone = pytz.timezone('Etc/GMT-3')
    now = datetime.now(timezone)
    schedule_time = now.replace(hour=0, minute=0, second=0, microsecond=0)

    # If it's already past midnight, schedule for the next day
    if now > schedule_time:
        schedule_time += timedelta(days=1)

    schedule.every().day.at(schedule_time.strftime("%H:%M")).do(news_fetch)

# Initial scheduling
schedule_job()

while True:
    schedule.run_pending()
    time.sleep(1)

In [ ]:
import pandas as pd
import schedule
import time
from datetime import datetime, timedelta
import pytz
import MetaTrader5 as mt5  # Ensure you have the MetaTrader5 package

def check_and_trade(stop_event, news_df):
    # Your trading logic here
    pass

def schedule_job():
    # Define your scheduling logic here
    pass

try:
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    news_df = pd.read_csv(file_name)

    # Start the trading loop
    check_and_trade(stop_event, news_df)

except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")

In [ ]:
def check_and_trade(stop_event, news_df):
    while not stop_event.is_set():
        try:
            # Initial scheduling
            schedule_job()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Ensure the 'date' column is treated as strings and strip spaces
            news_df['date'] = news_df['date'].astype(str).str.strip()

            # Parse the 'date' column with the correct format
            news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d', errors='coerce')

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue
                    
            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

In [ ]:
def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()

    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()

            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Filter news events for the current day
            current_date = MT5.date()
            news_df_today = news_df[news_df['date'].dt.date == current_date]

            # Check for upcoming news events
            news_times = news_df_today['time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
            for news_time in news_times:
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time}. Halting trading.")
                    time.sleep(60)
                    continue

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    time.sleep(0.1)
                    open_position = get_open_position()
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print("Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)

try:
    current_month = datetime.now().strftime('%B')
    file_name = f'{current_month}_news.csv'
    news_df = pd.read_csv(file_name)
    news_df['date'] = pd.to_datetime(news_df['date'])
    check_and_trade(stop_event, news_df)

except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    mt5.shutdown()
    print("MetaTrader 5 connection closed")

In [ ]:
import pandas as pd
import threading
from datetime import datetime, timedelta
import MetaTrader5 as mt5
import schedule
import time

# Define your stop event
stop_event = threading.Event()

def convert_time_to_mt5(df):
    # Define the time zones
    utc_plus2_tz = pytz.timezone('Etc/GMT-2')  # Assumed Forex Factory time zone
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time with a default date
        time_str = row['time']
        naive_time = datetime.strptime("2023-01-01 " + time_str, "%Y-%m-%d %I:%M%p")  # Convert to naive datetime
        
        # Localize to UTC+2
        localized_time = utc_plus2_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)
    return df

def news_fetch():
    # Get the current month name
    current_month = datetime.now().strftime('%B')
    
    # Construct the file name
    file_name = f'{current_month}_news.csv'
    
    # Read the news CSV file into a DataFrame
    ds = pd.read_csv(file_name, parse_dates=["date"], index_col=0)
    
    # Filter the news DataFrame
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    
    # Convert the times to MT5 time zone
    ds = convert_time_to_mt5(ds)
    
    return ds

def schedule_job():
    schedule.every().day.at("00:01").do(update_news_df)

def update_news_df():
    global news_df
    news_df = news_fetch()

def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()

    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()

            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Filter news events for the current day
            current_date = MT5.date()
            news_df_today = news_df[news_df['date'].dt.date == current_date]

            # Check for upcoming news events
            news_times = news_df_today['time'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
            for news_time in news_times:
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time}. Halting trading.")
                    time.sleep(60)
                    continue

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    time.sleep(0.1)
                    open_position = get_open_position()
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print("Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)

try:
    current_month = datetime.now().strftime('%B')
    file_name = f'{current_month}_news.csv'
    news_df = pd.read_csv(file_name)
    news_df['date'] = pd.to_datetime(news_df['date'])
    check_and_trade(stop_event, news_df)

except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    mt5.shutdown()
    print("MetaTrader 5 connection closed")


In [ ]:
def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()

    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()

            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%Y-%m-%d %H:%M:%S')
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue


            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    time.sleep(0.1)
                    open_position = get_open_position()
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print("Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)


In [ ]:
# Filter news events for the current day
current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

# Check for upcoming news events
for index, row in news_df_today.iterrows():
    news_time = datetime.strptime(row['time'], '%Y-%m-%d %H:%M:%S')
    if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
        print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
        time.sleep(60)
        continue


In [ ]:
def execute_trade(signal, qty):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")


def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()
    
    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Ensure the 'date' column is treated as strings and strip spaces
            news_df['date'] = news_df['date'].astype(str).str.strip()

            # Parse the 'date' column with the correct format
            news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d', errors='coerce')

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue

                    
            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

In [ ]:
def execute_trade(signal, qty, upper_band, lower_band):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
        sl = lower_band
        tp = upper_band
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
        sl = upper_band
        tp = lower_band
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}, SL: {sl}, TP: {tp}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")

def check_and_trade(stop_event, news_df, upper_band, lower_band):
    # Initial scheduling
    schedule_job()
    
    last_closed_position_hit_sl_tp = None
    
    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Ensure the 'date' column is treated as strings and strip spaces
            news_df['date'] = news_df['date'].astype(str).str.strip()

            # Parse the 'date' column with the correct format
            news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d', errors='coerce')

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue

                    
            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    if open_position.sl or open_position.tp:
                        last_closed_position_hit_sl_tp = open_position.type
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00, upper_band, lower_band)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    if last_closed_position_hit_sl_tp is None or \
                       (last_closed_position_hit_sl_tp == mt5.ORDER_TYPE_BUY and current_signal == -1) or \
                       (last_closed_position_hit_sl_tp == mt5.ORDER_TYPE_SELL and current_signal == 1):
                        execute_trade(current_signal, 1.00, upper_band, lower_band)
                        last_closed_position_hit_sl_tp = None
                    else:
                        print("Waiting for opposite direction position before opening new trade.")
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error


In [ ]:
def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

# Fetch open position
open_position = get_open_position()

if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)
                    if open_position.sl or open_position.tp:
                        last_closed_position_hit_sl_tp = open_position.type
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00, upper_band, lower_band)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    if last_closed_position_hit_sl_tp is None or \
                       (last_closed_position_hit_sl_tp == mt5.ORDER_TYPE_BUY and current_signal == -1) or \
                       (last_closed_position_hit_sl_tp == mt5.ORDER_TYPE_SELL and current_signal == 1):
                        execute_trade(current_signal, 1.00, upper_band, lower_band)
                        last_closed_position_hit_sl_tp = None
                    else:
                        print("Waiting for opposite direction position before opening new trade.")
                else:
                    print("No signal to act upon.")

In [ ]:
# Initialize variables to track last trade outcome and direction
last_trade_hit_sl_tp = False
last_trade_direction = None

def check_and_trade(stop_event, news_df):
    # Initial scheduling
    schedule_job()
    
    while not stop_event.is_set():
        try:
            # Run scheduled tasks
            schedule.run_pending()
            
            # Print MT5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Ensure the 'date' column is treated as strings and strip spaces
            news_df['date'] = news_df['date'].astype(str).str.strip()

            # Parse the 'date' column with the correct format
            news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d', errors='coerce')

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(1)  # Sleep briefly to avoid busy waiting

            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes=30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            # If there's an open position, handle it according to the signal
            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}. Previous was opposite, closing position.")
                    close_position(open_position)

                    # Check if position was closed due to hitting SL or TP
                    if last_position_hit_sl_tp(open_position):
                        last_trade_hit_sl_tp = True
                        last_trade_direction = open_position.type
                    else:
                        last_trade_hit_sl_tp = False
                    
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        if not (last_trade_hit_sl_tp and current_signal == last_trade_direction):
                            execute_trade(current_signal, 1.00)
                        else:
                            print("Skipping trade in the same direction after hitting SL/TP.")
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    if not (last_trade_hit_sl_tp and current_signal == last_trade_direction):
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Skipping trade in the same direction after hitting SL/TP.")
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error

def last_position_hit_sl_tp():
    # Fetch the history of deals
    deals = mt5.history_deals_get()

    if deals is None or len(deals) == 0:
        return False  # No deals found

    last_deal = deals[-1]  # Get the most recent deal
    
    # Ensure the deal type is a closing order
    if last_deal.type == mt5.ORDER_TYPE_BUY or last_deal.type == mt5.ORDER_TYPE_SELL:
        # Fetch the stop loss and take profit from the deal
        stop_loss = last_deal.sl
        take_profit = last_deal.tp

        # Compare the deal exit price with SL and TP
        if last_deal.price == stop_loss:
            return 'SL'
        elif last_deal.price == take_profit:
            return 'TP'
    
    return False

In [ ]:
            # Ensure the 'date' column is treated as strings and strip spaces
            news_df['date'] = news_df['date'].astype(str).str.strip()

            # Add the current year to the 'date' column
            current_year = datetime.now().year
            news_df['date'] = news_df['date'] + f' {current_year}'

            # Parse the 'date' column with the correct format
            news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d %Y', errors='coerce')

            # Filter news events for the current day
            current_date = MT5.strftime('%b %d')  # Format current date as "Sep 23"
            news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

            print(f"Current date: {current_date}")
            print(f"News events for today:\n{news_df_today}")

            # Check for upcoming news events
            for index, row in news_df_today.iterrows():
                news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
                news_time = news_time.replace(year=current_year, month=MT5.month, day=MT5.day)  # Ensure the correct date
                print(f"Checking news event at {news_time} with current MT5 time {MT5}")
                if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
                    # Fetch open position
                    open_position = get_open_position()
                    if open_position:
                        close_position(open_position)
                        print("Closed open position due to upcoming news event.")
                    print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
                    time.sleep(60)
                    continue


In [ ]:
# Ensure the 'date' column is treated as strings and strip spaces
news_df['date'] = news_df['date'].astype(str).str.strip()

# Add the current year to the 'date' column
current_year = datetime.now().year
news_df['date'] = news_df['date'] + f' {current_year}'

# Parse the 'date' column with the correct format
news_df['date'] = pd.to_datetime(news_df['date'], format='%b %d %Y', errors='coerce')

# Filter news events for the current day
current_date = MT5.strftime('%b %d')  # Format current date as "Sep 24"
news_df_today = news_df[news_df['date'].dt.strftime('%b %d') == current_date]

print(f"Current date: {current_date}")
print(f"News events for today:\n{news_df_today}")

# Check for upcoming news events
for index, row in news_df_today.iterrows():
    news_time = datetime.strptime(row['time'], '%I:%M %p')  # Adjusted format to match '04:45 PM'
    news_time = news_time.replace(year=current_year, month=MT5.month, day=MT5.day)  # Ensure the correct date
    print(f"Checking news event at {news_time} with current MT5 time {MT5}")
    if news_time - timedelta(minutes=15) <= MT5 <= news_time + timedelta(minutes=15):
        # Fetch open position
        open_position = get_open_position()
        if open_position:
            close_position(open_position)
            print("Closed open position due to upcoming news event.")
        print(f"News event at {news_time} and the news is {row['event']}. Halting trading.")
        time.sleep(60)
        continue


In [ ]:
import logging
import MetaTrader5 as mt5

# Configure logging
logging.basicConfig(filename='trading_journal.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

def execute_trade(signal, qty, upper, lower):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
        sl = lower
        tp = upper
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
        sl = upper
        tp = lower
    else:
        logging.info("No action needed")
        return
    
    logging.info(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "sl": sl,
        "tp": tp,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    logging.info(f"Trade order result: {result}")

def modify_open_position(upper, lower):
    if not mt5.initialize():
        logging.error("initialize() failed")
        mt5.shutdown()
        return None

    positions = mt5.positions_get()
    if positions is None or len(positions) == 0:
        logging.info("No open positions found")
        mt5.shutdown()
        return None

    position = positions[0]

    if position.type == mt5.ORDER_TYPE_BUY:
        sl = lower
        tp = upper
    elif position.type == mt5.ORDER_TYPE_SELL:
        sl = upper
        tp = lower
    else:
        logging.error("Unsupported position type")
        mt5.shutdown()
        return None

    request = {
        "action": mt5.TRADE_ACTION_SLTP,
        "symbol": position.symbol,
        "sl": sl,
        "tp": tp,
        "position": position.ticket,
    }

    result = mt5.order_send(request)
    logging.info(f"Modify position result: {result}")

    mt5.shutdown()

# Example usage
upper = 1.2400
lower = 1.2300
signal = 1
qty = 0.1

execute_trade(signal, qty, upper, lower)
modify_open_position(upper, lower)


In [ ]:
def check_last_closed_order():

    # Get the history of closed orders
    from_date = mt5.copy_rates_from_pos("EURUSD", mt5.TIMEFRAME_M1, 0, 1)[0]['time']
    to_date = mt5.copy_rates_from_pos("EURUSD", mt5.TIMEFRAME_M1, 0, 1)[0]['time']
    closed_orders = mt5.history_orders_get(from_date, to_date)
    
    if closed_orders is None or len(closed_orders) == 0:
        print("No closed orders found")
        return None

    # Get the last closed order
    last_order = closed_orders[-1]

    # Extract relevant details
    close_price = last_order.price_current
    sl = last_order.sl
    tp = last_order.tp
    order_type = last_order.type

    # Determine if the order hit SL or TP
    if order_type == mt5.ORDER_TYPE_BUY:
        if close_price <= sl:
            result = "Hit Stop Loss"
        elif close_price >= tp:
            result = "Hit Take Profit"
        else:
            result = "Closed manually or other reason"
    elif order_type == mt5.ORDER_TYPE_SELL:
        if close_price >= sl:
            result = "Hit Stop Loss"
        elif close_price <= tp:
            result = "Hit Take Profit"
        else:
            result = "Closed manually or other reason"
    else:
        result = "Unknown order type"
    return result

# Example usage
result = check_last_closed_order()
print(f"Last closed order result: {result}")


In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Email configuration
SMTP_SERVER = 'smtp.office365.com'
SMTP_PORT = 587
EMAIL_ADDRESS = os.environ['EMAIL_ADDRESS']
EMAIL_PASSWORD = os.environ['EMAIL_PASSWORD']

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = os.environ['EMAIL_ADDRESS']
    msg['To'] = os.environ['ALERT_RECIPIENT']
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_ADDRESS, EMAIL_ADDRESS, text)
        server.quit()
        logging.info('Email sent successfully')
    except Exception as e:
        logging.error(f'Failed to send email: {e}')

# Example function to open a position
def open_position(position_details):
    # Your logic to open a position
    logging.info(f'Position opened: {position_details}')
    send_email('New Position Opened', f'Position details: {position_details}')



In [ ]:
import msal
import requests

# Define your application credentials
CLIENT_ID = 'your-client-id'
CLIENT_SECRET = 'your-client-secret'
TENANT_ID = 'your-tenant-id'
AUTHORITY = f'https://login.microsoftonline.com/{TENANT_ID}'
SCOPES = ['https://graph.microsoft.com/.default']

# Create a confidential client application
app = msal.ConfidentialClientApplication(
    CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET
)

# Acquire a token
result = app.acquire_token_for_client(scopes=SCOPES)

if 'access_token' in result:
    access_token = result['access_token']
    # Use the access token to send an email
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    email_data = {
        'message': {
            'subject': 'New Position Opened',
            'body': {
                'contentType': 'Text',
                'content': 'Position details: ...'
            },
            'toRecipients': [
                {
                    'emailAddress': {
                        'address': 'recipient@example.com'
                    }
                }
            ]
        }
    }
    response = requests.post(
        'https://graph.microsoft.com/v1.0/me/sendMail',
        headers=headers,
        json=email_data
    )
    if response.status_code == 202:
        logging.info('Email sent successfully')
    else:
        logging.error(f'Failed to send email: {response.status_code}')
else:
    logging.error('Failed to acquire token')


In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import logging

# Email configuration
SMTP_SERVER = 'smtp.office365.com'
SMTP_PORT = 587
EMAIL_ADDRESS = os.environ['EMAIL_ADDRESS']
EMAIL_PASSWORD = 'your_password_here'  # Replace with your actual password

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = os.environ['ALERT_RECIPIENT']  # Intended recipient
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_ADDRESS, msg['To'], text)
        server.quit()
        logging.info('Email sent successfully')
    except Exception as e:
        logging.error(f'Failed to send email: {e}')

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587
EMAIL_ADDRESS = 'your_gmail@gmail.com'
EMAIL_PASSWORD = 'your_app_password'  # Use the 16-character app password

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = 'recipient@example.com'
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, msg['To'], msg.as_string())
        server.quit()
        print("Email sent successfully!")
    except Exception as e:
        print(f"Failed to send email: {e}")

# Example usage
send_email('Trading Status', 'Your latest trading update is here.')

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import logging

# Email configuration (Zoho Mail settings)
SMTP_SERVER = 'smtp.zoho.com'
SMTP_PORT = 587  # or 465 for SSL
EMAIL_ADDRESS = 'your_zoho_email@zoho.com'
EMAIL_PASSWORD = 'your_zoho_password'

def send_email(subject, body):
    msg = MIMEMultipart()
    msg['From'] = EMAIL_ADDRESS
    msg['To'] = 'recipient@example.com'  # Intended recipient
    msg['Subject'] = subject

    msg.attach(MIMEText(body, 'plain'))

    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()  # Start TLS for security
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_ADDRESS, msg['To'], text)
        server.quit()
        logging.info('Email sent successfully')
    except Exception as e:
        logging.error(f'Failed to send email: {e}')

# Example usage
send_email('Test Subject', 'This is a test email body.')